# Practice #6. "Neural Networks for Time Series Forecasting"

This notebook is dedicated to:
* Neural Network Fundamentals for Time Series
* Multi-layer Perceptron (MLP) for Time Series
* Recurrent Neural Networks (RNN/LSTM)
* PyTorch Implementation and Training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 0. Data reading and visualization

Please, specify path to data

In [ ]:
path_to_datafile = "../data/daily-total-female-births.csv"

In [ ]:
# data reading to pandas.DataFrame
df = pd.read_csv(path_to_datafile)

Please, rename time column to `ds` and data column to `y`(you can use `df.rename`) . If use dataset with multiple features select only one and drop NaN values

In [ ]:
# your code here
# df.rename(columns={"...": "ds", "...": "y"}, inplace=True)

Convert date column to datetime format and set as index

In [ ]:
df["ds"] = pd.to_datetime(df["ds"])
df.set_index("ds", inplace=True)

Let's plot the data

In [ ]:
plt.figure(figsize=(20, 5))
plt.ylabel("y")
plt.xlabel("ds")
plt.plot(df);

## 1. Data Preparation for Neural Networks

Neural networks require careful data preparation:
1. **Normalization**: Scale data to improve training stability
2. **Sequence Creation**: Create input-output sequences for supervised learning
3. **Train/Validation/Test Split**: Proper splitting for time series
4. **Tensor Conversion**: Convert to PyTorch tensors

### 1.1 Data Normalization

In [ ]:
# your code here
# Normalize the data
# ...


### 1.2 Create Sequences for Supervised Learning

In [ ]:
def create_sequences(data, seq_length, pred_length=1):
    """
    Create sequences for time series forecasting
    
    Args:
        data: 1D array of time series data
        seq_length: Length of input sequence (lookback window)
        pred_length: Length of prediction horizon
    
    Returns:
        X: Input sequences of shape (n_samples, seq_length)
        y: Target sequences of shape (n_samples, pred_length)
    """
    X, y = [], []
    
    for i in range(len(data) - seq_length - pred_length + 1):
        # Input sequence
        seq_x = data[i:(i + seq_length)]
        # Target sequence
        seq_y = data[(i + seq_length):(i + seq_length + pred_length)]
        
        X.append(seq_x)
        y.append(seq_y)
    
    return np.array(X), np.array(y)

# your code here
# Create sequences
# seq_length = ...

# print(f"Input sequences shape: {X.shape}")
# print(f"Target sequences shape: {y.shape}")
# print(f"Example input sequence: {X[0]}")
# print(f"Example target: {y[0]}")

### 1.3 Train/Validation/Test Split

In [ ]:
# your code here
# Split data maintaining temporal order
# ...

# print(f"Training set: {X_train.shape[0]} samples")
# print(f"Validation set: {X_val.shape[0]} samples")
# print(f"Test set: {X_test.shape[0]} samples")

# Convert to PyTorch tensors
# ...

## 2. Multi-layer Perceptron (MLP) for Time Series

A Multi-layer Perceptron treats time series forecasting as a standard regression problem. The input sequence is flattened and fed through fully connected layers.

**Architecture:**
```
Input (seq_length,) → Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear → Output
```

### 2.1 MLP Model Definition

In [ ]:
class MLPForecaster(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout_rate=0.2):
        super(MLPForecaster, self).__init__()
        
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        
        # Hidden layers
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
        
        # Output layer
        layers.append(nn.Linear(hidden_sizes[-1], output_size))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# your code here
# Define MLP model
# ...

# mlp_model = ...
# print(mlp_model)

# Count parameters
# total_params = ...
# print(f"Total trainable parameters: {total_params}")

### 2.2 Training Functions

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.001, batch_size=32):
    """
    Train a PyTorch model with validation monitoring
    """
    # Create data loaders
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    # Training history
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
        
        # Calculate average losses
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Print progress
        if (epoch + 1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    return train_losses, val_losses

def plot_training_history(train_losses, val_losses, title):
    """
    Plot training and validation loss curves
    """
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'{title} - Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_losses[-50:], label='Training Loss (Last 50 epochs)')
    plt.plot(val_losses[-50:], label='Validation Loss (Last 50 epochs)')
    plt.title('Loss Curves (Zoomed)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

### 2.3 Train MLP Model

In [ ]:
# your code here
# Train MLP model
# print("Training MLP model...")
# ...


## 3. Recurrent Neural Networks (RNN/LSTM)

Recurrent Neural Networks are specifically designed for sequential data. LSTM (Long Short-Term Memory) networks can capture long-term dependencies in time series.

**Key Advantages:**
- Can process sequences of varying lengths
- Maintain internal state (memory)
- Better at capturing temporal patterns
- LSTM solves vanishing gradient problem of basic RNNs

### 3.1 LSTM Model Definition

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout_rate=0.2):
        super(LSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout_rate if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Output layer
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(x, (h0, c0))
        
        # Take only the last output
        last_output = lstm_out[:, -1, :]
        
        # Apply dropout and linear layer
        output = self.dropout(last_output)
        output = self.linear(output)
        
        return output

# your code here
# Reshape data for LSTM (batch_size, seq_length, features)
# ...

# print(f"LSTM input shape: {X_train_lstm.shape}")

# Define LSTM model
# ...

# print(lstm_model)

# Count parameters
# total_params = ...
# print(f"Total trainable parameters: {total_params}")

### 3.2 Train LSTM Model

In [ ]:
# your code here
# Train LSTM model
# print("Training LSTM model...")
# ...

### 3.3 Bidirectional LSTM

In [ ]:
class BiLSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout_rate=0.2):
        super(BiLSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout_rate if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True
        )
        
        # Output layer (hidden_size * 2 because bidirectional)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(hidden_size * 2, output_size)
    
    def forward(self, x):
        # Initialize hidden state (num_layers * 2 for bidirectional)
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM forward pass
        lstm_out, _ = self.lstm(x, (h0, c0))
        
        # Take only the last output
        last_output = lstm_out[:, -1, :]
        
        # Apply dropout and linear layer
        output = self.dropout(last_output)
        output = self.linear(output)
        
        return output

# your code here
# Define Bidirectional LSTM model
# ...

# print(bilstm_model)

# Train Bidirectional LSTM model
# ...

### 4.5 Questions for Analysis

**Questions to consider:**

1. **Model Performance:**
   - Which neural network architecture performed best and why?
   - How do neural networks compare to classical time series methods?
   - What are the trade-offs between model complexity and performance?

2. **Architecture Comparison:**
   - Why might LSTM perform better than MLP for time series?
   - What are the advantages of bidirectional LSTM?
   - How does the number of parameters affect performance?

3. **Training Dynamics:**
   - How do the loss curves compare between models?
   - Are there signs of overfitting or underfitting?
   - How effective is the learning rate scheduling?

4. **Practical Considerations:**
   - What is the computational cost vs. accuracy trade-off?
   - How would you deploy these models in production?
   - What hyperparameters would you tune further?

**Potential Improvements:**
- Implement attention mechanisms
- Try different sequence lengths
- Experiment with multi-step forecasting
- Add feature engineering (external variables)
- Implement ensemble methods
- Use advanced architectures (Transformer, GRU)
- Apply techniques like teacher forcing for training